In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt

In [2]:
# Dataset 1: Portuguese student performance (semicolon-separated)
df1 = pd.read_csv("/Users/pari/Downloads/student-por.csv", sep=";")


# Dataset 2: Higher education students (comma-separated
df2 = pd.read_csv("/Users/pari/Downloads/DATA (1).csv")

print("Dataset 1 shape:", df1.shape)
print("Dataset 2 shape:", df2.shape)
df1.head()


Dataset 1 shape: (649, 33)
Dataset 2 shape: (145, 33)


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,4,0,11,11
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,2,9,11,11
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,6,12,13,12
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,0,14,14,14
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,0,11,13,13


In [3]:

d1 = df1.copy()

# Turn the final grade G3 into a pass/fail target (>=10 is pass in Portuguese grading)
d1["pass"] = (d1["G3"] >= 10).astype(int)

# Drop the grade columns so the model predicts from student features, not from earlier grades
d1 = d1.drop(columns=["G1", "G2", "G3"])

# Convert text columns (e.g. 'GP', 'F', 'yes') into numbers using label encoding
for col in d1.select_dtypes(include="object").columns:
    d1[col] = LabelEncoder().fit_transform(d1[col])

# Separate features (X) from the target (y)
X1 = d1.drop(columns=["pass"])
y1 = d1["pass"]

print("Dataset 1 ready. Features:", X1.shape, "Target distribution:")
print(y1.value_counts())

Dataset 1 ready. Features: (649, 30) Target distribution:
pass
1    549
0    100
Name: count, dtype: int64


In [4]:
d2 = df2.copy()

# Drop identifier columns that aren't useful features
d2 = d2.drop(columns=["STUDENT ID", "COURSE ID"])

# Turn the multi-class GRADE (0-7) into pass/fail: 0 = Fail, everything else = pass
d2["pass"] = (d2["GRADE"] > 0).astype(int)
d2 = d2.drop(columns=["GRADE"])

# All columns are already numeric here, so no encoding needed
# Separate features (X) from target (y)
X2 = d2.drop(columns=["pass"])
y2 = d2["pass"]

print("Dataset 2 ready. Features:", X2.shape, "Target distribution:")
print(y2.value_counts())

Dataset 2 ready. Features: (145, 30) Target distribution:
pass
1    137
0      8
Name: count, dtype: int64


In [5]:
# Split into training and test sets (80% train, 20% test)
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, random_state=42, stratify=y2)

# --- Model 1: Decision Tree ---
dt2 = DecisionTreeClassifier(random_state=42)
dt2.fit(X2_train, y2_train)
dt2_pred = dt2.predict(X2_test)

# --- Model 2: Random Forest ---
rf2 = RandomForestClassifier(random_state=42)
rf2.fit(X2_train, y2_train)
rf2_pred = rf2.predict(X2_test)

# --- Results ---
print("=== DATASET 2: DECISION TREE ===")
print("Accuracy:", round(accuracy_score(y2_test, dt2_pred), 3))
print(classification_report(y2_test, dt2_pred))

print("=== DATASET 2: RANDOM FOREST ===")
print("Accuracy:", round(accuracy_score(y2_test, rf2_pred), 3))
print(classification_report(y2_test, rf2_pred))

=== DATASET 2: DECISION TREE ===
Accuracy: 0.966
              precision    recall  f1-score   support

           0       0.67      1.00      0.80         2
           1       1.00      0.96      0.98        27

    accuracy                           0.97        29
   macro avg       0.83      0.98      0.89        29
weighted avg       0.98      0.97      0.97        29

=== DATASET 2: RANDOM FOREST ===
Accuracy: 0.931
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         2
           1       0.93      1.00      0.96        27

    accuracy                           0.93        29
   macro avg       0.47      0.50      0.48        29
weighted avg       0.87      0.93      0.90        29



/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [6]:
# Split into training and test sets (80% train, 20% test)
X1_train, X1_test, y1_train, y1_test = train_test_split(
    X1, y1, test_size=0.2, random_state=42, stratify=y1)

# --- Model 1: Decision Tree ---
dt1 = DecisionTreeClassifier(random_state=42)
dt1.fit(X1_train, y1_train)
dt1_pred = dt1.predict(X1_test)

# --- Model 2: Random Forest ---
rf1 = RandomForestClassifier(random_state=42)
rf1.fit(X1_train, y1_train)
rf1_pred = rf1.predict(X1_test)

# --- Results ---
print("=== DATASET 1: DECISION TREE ===")
print("Accuracy:", round(accuracy_score(y1_test, dt1_pred), 3))
print(classification_report(y1_test, dt1_pred))

print("=== DATASET 1: RANDOM FOREST ===")
print("Accuracy:", round(accuracy_score(y1_test, rf1_pred), 3))
print(classification_report(y1_test, rf1_pred))

=== DATASET 1: DECISION TREE ===
Accuracy: 0.754
              precision    recall  f1-score   support

           0       0.20      0.20      0.20        20
           1       0.85      0.85      0.85       110

    accuracy                           0.75       130
   macro avg       0.53      0.53      0.53       130
weighted avg       0.75      0.75      0.75       130

=== DATASET 1: RANDOM FOREST ===
Accuracy: 0.8
              precision    recall  f1-score   support

           0       0.20      0.10      0.13        20
           1       0.85      0.93      0.89       110

    accuracy                           0.80       130
   macro avg       0.53      0.51      0.51       130
weighted avg       0.75      0.80      0.77       130

